# Prepare LoRA Data

Convert benchmark records into the chat JSONL format expected by `mlx_lm.lora`.

In [ ]:
from pathlib import Path
import json
import sys

In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
from training_eval.eval_utils import load_jsonl_records

In [ ]:
SYSTEM_MESSAGE = "You solve discrete stochastic-process problems. Give the reasoning, then put the final JSON answer inside <answer>...</answer>."

In [ ]:
DATASETS = {
    "explicit_theorems": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val",
    },
    "implicit_theorems": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_implicit_theorems",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_implicit_theorems",
    },
}

OUTPUT_ROOT = PROJECT_ROOT / "training_eval" / "fine_tune_qwen1_7B" / "lora" / "data"

In [ ]:
def make_chat_example(record):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": record["problem"]},
            {"role": "assistant", "content": record["reasoning"]},
        ]
    }

In [ ]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for record in records:
            f.write(json.dumps(record, sort_keys=True) + "\n")

In [ ]:
def build_lora_dataset(name, paths):
    train_records = load_jsonl_records(paths["train"])
    valid_records = load_jsonl_records(paths["valid"])
    output_dir = OUTPUT_ROOT / name

    write_jsonl([make_chat_example(record) for record in train_records], output_dir / "train.jsonl")
    write_jsonl([make_chat_example(record) for record in valid_records], output_dir / "valid.jsonl")

    return {
        "name": name,
        "output_dir": output_dir,
        "train_records": len(train_records),
        "valid_records": len(valid_records),
    }

In [ ]:
summaries = [build_lora_dataset(name, paths) for name, paths in DATASETS.items()]
summaries

In [ ]:
example_path = OUTPUT_ROOT / "explicit_theorems" / "train.jsonl"
json.loads(example_path.read_text().splitlines()[0])